# 🎓 Fine-Tuning Experiments - TripTrove

Notebook untuk eksperimen dengan fine-tuning techniques.

## Tujuan:
- Test Few-Shot Learning effectiveness
- Prepare data for LoRA fine-tuning
- Compare before/after fine-tuning
- Measure improvement metrics

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / 'src'
fine_tuning_path = project_root / 'fine_tuning'

sys.path.insert(0, str(src_path))
sys.path.insert(0, str(fine_tuning_path))

print(f"✅ Paths configured")

In [ ]:
# Imports
from agent_rag import TripTroveAgent
from few_shot_examples import FEW_SHOT_EXAMPLES
import json
import time

print("✅ Libraries imported!")

## 1. Test Few-Shot Learning

In [ ]:
# View few-shot examples
print(f"📚 Total Few-Shot Examples: {len(FEW_SHOT_EXAMPLES)}\n")

for i, example in enumerate(FEW_SHOT_EXAMPLES, 1):
    print(f"Example {i}:")
    print(f"Q: {example['question']}")
    print(f"A: {example['answer'][:100]}...")
    print("-" * 80)

In [ ]:
# Initialize agent (with few-shot learning)
print("🔄 Initializing agent with Few-Shot Learning...")
agent = TripTroveAgent()
print("✅ Agent initialized!")

In [ ]:
# Test queries similar to few-shot examples
test_queries = [
    "Berapa harga paket tour ke Bali?",
    "Apa saja term and condition TripTrove?",
    "Rekomendasi paket tour untuk keluarga?",
    "Paket tour adventure apa yang tersedia?"
]

print("\n🧪 Testing Few-Shot Learning Effect:\n")

for query in test_queries:
    print(f"{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    response = agent.query(query)
    print(f"\nResponse:\n{response}\n")

## 2. Prepare Training Data for LoRA

In [ ]:
# Run training data preparation
import subprocess

print("🔄 Generating training data...")
result = subprocess.run(
    ['python', str(fine_tuning_path / 'prepare_training_data.py')],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode == 0:
    print("✅ Training data generated!")
else:
    print(f"❌ Error: {result.stderr}")

In [ ]:
# Load and inspect training data
training_file = fine_tuning_path / 'training_data.jsonl'

if training_file.exists():
    with open(training_file, 'r', encoding='utf-8') as f:
        training_data = [json.loads(line) for line in f]
    
    print(f"📊 Training Data Statistics:")
    print(f"Total examples: {len(training_data)}")
    
    print(f"\n📝 Sample Training Examples:\n")
    for i, example in enumerate(training_data[:3], 1):
        print(f"Example {i}:")
        print(f"Prompt: {example['prompt'][:100]}...")
        print(f"Response: {example['response'][:100]}...")
        print("-" * 80)
else:
    print("⚠️ Training data file not found")

## 3. Compare Response Quality

In [ ]:
# Define evaluation criteria
def evaluate_response(response, query):
    """
    Simple evaluation metrics
    """
    metrics = {
        'length': len(response),
        'has_price': 'Rp' in response or 'harga' in response.lower(),
        'has_details': len(response.split()) > 20,
        'is_helpful': any(word in response.lower() for word in ['tersedia', 'dapat', 'bisa', 'ada']),
        'is_polite': any(word in response.lower() for word in ['terima kasih', 'silakan', 'senang'])
    }
    
    score = sum([
        metrics['has_price'] * 2,
        metrics['has_details'] * 2,
        metrics['is_helpful'] * 1,
        metrics['is_polite'] * 1
    ])
    
    return metrics, score

print("✅ Evaluation function ready")

In [ ]:
# Evaluate responses
evaluation_queries = [
    "Berapa harga paket tour ke Bali?",
    "Rekomendasi paket tour untuk honeymoon?",
    "Apa kebijakan pembatalan?"
]

results = []

for query in evaluation_queries:
    print(f"\n{'='*80}")
    print(f"Evaluating: {query}")
    print(f"{'='*80}")
    
    response = agent.query(query)
    metrics, score = evaluate_response(response, query)
    
    print(f"\nResponse:\n{response}")
    print(f"\n📊 Metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")
    print(f"\n⭐ Score: {score}/6")
    
    results.append({
        'query': query,
        'response': response,
        'metrics': metrics,
        'score': score
    })

In [ ]:
# Summary
import pandas as pd

df_results = pd.DataFrame([{
    'query': r['query'],
    'score': r['score'],
    'length': r['metrics']['length']
} for r in results])

print("\n📊 Evaluation Summary:")
print(df_results)
print(f"\nAverage Score: {df_results['score'].mean():.2f}/6")

## 4. LoRA Fine-Tuning Instructions

### To run LoRA fine-tuning:

```bash
# From project root
cd fine_tuning
python fine_tune_lora.py
```

### Requirements:
- GPU with at least 8GB VRAM (recommended)
- Training will take 1-2 hours depending on hardware
- Output model will be saved to `fine_tuning/triptrove-llama-lora/`

### After Fine-Tuning:
1. Test the fine-tuned model
2. Compare with base model
3. Measure improvement in response quality
4. Deploy if results are satisfactory

## 5. Add Custom Few-Shot Examples

In [ ]:
# Create custom few-shot example
custom_example = {
    'question': input("Enter example question: "),
    'answer': input("Enter ideal answer: ")
}

if custom_example['question'] and custom_example['answer']:
    print("\n✅ Custom example created:")
    print(f"Q: {custom_example['question']}")
    print(f"A: {custom_example['answer']}")
    print("\n💡 Add this to fine_tuning/few_shot_examples.py to use it!")

## 📝 Fine-Tuning Best Practices

1. **Few-Shot Learning** (Current):
   - Quick to implement
   - No training required
   - Good for small improvements
   - Limited by context window

2. **LoRA Fine-Tuning** (Advanced):
   - Requires GPU and time
   - Better long-term results
   - Model learns patterns deeply
   - Recommended for production

3. **Evaluation**:
   - Always compare before/after
   - Use consistent test queries
   - Measure multiple metrics
   - Get user feedback